# Pelanca NNUE v2 — Treino Forte (~100M posicoes)

**Mudancas vs v1:**
- **~100M posicoes brutas** via wget direto (5 parquets, ~2 min download)
- Filtro depth ≥ 12 → ~20-25M posicoes limpas
- Arquitetura: 768→256→**8**→1 (forward 4x mais rapido)
- Loss **WDL-blended** (70% eval + 30% resultado do jogo)
- Filtros: skip check, evals extremos, jogos curtos
- **60 epochs**, cosine annealing LR

**Kaggle:** GPU T4 x2 ou P100, Internet ON.

In [ ]:
!pip install python-chess -q 2>/dev/null
!pip install h5py tqdm -q 2>/dev/null

import chess
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import h5py
import struct
import os
import time
from tqdm.auto import tqdm

print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# ============================================================
# CONFIGURACAO v2 — MAXIMIZADA
# ============================================================

MIN_DEPTH = 12
MAX_ABS_CP = 3000
EVAL_SCALE = 400.0
HDF5_FILE = '/kaggle/working/positions_v2.h5'
RAW_DIR = '/kaggle/working/parquets'
os.makedirs(RAW_DIR, exist_ok=True)

PARQUET_IDS = ['00015', '00016']
BASE_URL = "https://huggingface.co/datasets/Lichess/chess-position-evaluations/resolve/main/data"

# Arquitetura
INPUT_SIZE = 768
FT_SIZE = 256
HIDDEN_SIZE = 8

# Treino — GPU MAXIMIZADA
EPOCHS = 80
BATCH_SIZE = 524288             # 16x original (512K samples por batch)
LR_MAX = 4e-3                  # linear scaling: 16x batch -> 4x LR
LR_MIN = 1e-5
WEIGHT_DECAY = 1e-6
WDL_WEIGHT = 0.3
MIRROR_AUG = True

# Quantizacao
FT_QUANT = 64
HIDDEN_QUANT = 64
OUTPUT_QUANT = 64

# Output
NNUE_OUTPUT = '/kaggle/working/pelanca_v2.nnue'
CKPT_DIR = '/kaggle/working/checkpoints_v2'
os.makedirs(CKPT_DIR, exist_ok=True)

params = INPUT_SIZE*FT_SIZE + FT_SIZE + FT_SIZE*2*HIDDEN_SIZE + HIDDEN_SIZE + HIDDEN_SIZE + 1
print(f'Arch: {INPUT_SIZE}->{FT_SIZE}->{HIDDEN_SIZE}->1 ({params:,} params)')
print(f'Mirror: {MIRROR_AUG} | Batch: {BATCH_SIZE:,} | Epochs: {EPOCHS}')

---
## Download Direto (wget ~2 min) + Filtro com Pandas

In [ ]:
if os.path.exists(HDF5_FILE):
    size_mb = os.path.getsize(HDF5_FILE) / 1024**2
    print(f'Dataset existe: {HDF5_FILE} ({size_mb:.0f} MB)')
else:
    t0 = time.time()

    # === 1. DOWNLOAD silencioso ===
    for pid in PARQUET_IDS:
        fname = f'train-{pid}-of-00017.parquet'
        fpath = os.path.join(RAW_DIR, fname)
        if os.path.exists(fpath):
            sz = os.path.getsize(fpath) / 1024**2
            if sz > 100:  # arquivo valido (>100MB)
                print(f'  {fname} ja existe ({sz:.0f} MB)')
                continue
            else:
                os.remove(fpath)  # corrompido, re-baixar
        url = f'{BASE_URL}/{fname}?download=true'
        print(f'  Baixando {fname}...', end=' ', flush=True)
        ret = os.system(f'wget -q -O "{fpath}" "{url}" 2>/dev/null')
        sz = os.path.getsize(fpath) / 1024**2 if os.path.exists(fpath) else 0
        print(f'{sz:.0f} MB' if ret == 0 else 'ERRO!')

    dl_time = time.time() - t0
    print(f'\nDownload: {dl_time/60:.1f} min\n')

    # === 2. FILTRAR 100% VECTORIZADO (sem iterrows, sem chess.Board) ===
    # ~2 min para 50M posicoes em vez de 2h
    all_fens = []
    all_evals = []
    total_raw = 0

    for pid in PARQUET_IDS:
        fname = f'train-{pid}-of-00017.parquet'
        fpath = os.path.join(RAW_DIR, fname)
        print(f'Processando {fname}...')
        t1 = time.time()

        df = pd.read_parquet(fpath, columns=['fen', 'depth', 'cp', 'mate'])
        total_raw += len(df)
        print(f'  Bruto: {len(df):,}')

        # Filtro depth (vectorizado)
        df = df[df['depth'] >= MIN_DEPTH]

        # Filtro cp extremo (vectorizado)
        mask_cp = df['cp'].notna() & (df['cp'].abs() <= MAX_ABS_CP)
        mask_mate = df['mate'].notna() & (df['mate'].abs() > 3)
        df = df[mask_cp | mask_mate]

        # Dedup (vectorizado)
        df = df.drop_duplicates(subset='fen')

        # Calcular eval (vectorizado - sem loop!)
        evals = np.zeros(len(df), dtype=np.float32)
        cp_mask = df['cp'].notna().values
        mate_mask = df['mate'].notna().values
        cp_vals = df['cp'].fillna(0).values.astype(np.float64)
        mate_vals = df['mate'].fillna(0).values.astype(np.float64)

        evals[cp_mask] = np.tanh(cp_vals[cp_mask] / EVAL_SCALE).astype(np.float32)
        evals[mate_mask & (mate_vals > 0)] = 1.0
        evals[mate_mask & (mate_vals < 0)] = -1.0

        # Validar FEN por string (sem chess.Board!)
        # FEN valido: tem exatamente 1 rei branco e 1 preto, 6 partes
        fens = df['fen'].values
        valid = np.ones(len(fens), dtype=bool)
        for i in range(len(fens)):
            fen = fens[i]
            board_part = fen.split(' ')[0] if ' ' in fen else fen
            # Deve ter exatamente 1 K e 1 k
            if board_part.count('K') != 1 or board_part.count('k') != 1:
                valid[i] = False

        fens = fens[valid]
        evals = evals[valid]

        all_fens.extend(fens.tolist())
        all_evals.extend(evals.tolist())

        elapsed = time.time() - t1
        print(f'  Filtrado: {len(fens):,} posicoes em {elapsed:.0f}s')

        del df, fens, evals, valid
        import gc; gc.collect()

    # Dedup global (entre parquets)
    print(f'\nDedup global...')
    seen = set()
    unique_fens = []
    unique_evals = []
    for f, e in zip(all_fens, all_evals):
        if f not in seen:
            seen.add(f)
            unique_fens.append(f)
            unique_evals.append(e)

    total_kept = len(unique_fens)
    print(f'Total unico: {total_kept:,} de {total_raw:,} ({total_kept/total_raw*100:.0f}%)')

    del all_fens, all_evals, seen
    gc.collect()

    # === 3. SHUFFLE + SPLIT + SALVAR HDF5 ===
    rng = np.random.default_rng(42)
    indices = rng.permutation(total_kept)
    val_n = int(total_kept * 0.03)
    val_idx, train_idx = indices[:val_n], indices[val_n:]

    fens_arr = np.array(unique_fens, dtype=object)
    evals_arr = np.array(unique_evals, dtype=np.float32)
    # WDL = eval como proxy (dataset nao tem resultado do jogo)
    wdl_arr = evals_arr.copy()

    dt_str = h5py.string_dtype()
    with h5py.File(HDF5_FILE, 'w') as f:
        for name, idx in [('train', train_idx), ('val', val_idx)]:
            g = f.create_group(name)
            g.create_dataset('fens', data=fens_arr[idx], dtype=dt_str)
            g.create_dataset('evals', data=evals_arr[idx], compression='gzip')
            g.create_dataset('wdl', data=wdl_arr[idx], compression='gzip')
        f.attrs['total'] = total_kept
        f.attrs['min_depth'] = MIN_DEPTH

    elapsed = time.time() - t0
    size_mb = os.path.getsize(HDF5_FILE) / 1024**2
    print(f'\nSalvo: {HDF5_FILE} ({size_mb:.0f} MB) em {elapsed/60:.1f} min')
    print(f'  Train: {len(train_idx):,} | Val: {len(val_idx):,}')

    # Limpar parquets
    for pid in PARQUET_IDS:
        p = os.path.join(RAW_DIR, f'train-{pid}-of-00017.parquet')
        if os.path.exists(p): os.remove(p)
    print('Parquets removidos')

    del unique_fens, unique_evals, fens_arr, evals_arr, wdl_arr
    import gc; gc.collect()

---
## Feature Encoding (parsing rapido sem chess.Board) & Dataset (packed arrays)

Duas otimizacoes criticas para caber 44M posicoes em 30GB RAM:
1. **fen_to_sparse_fast**: parseia FEN com string ops em vez de chess.Board (~15x mais rapido)
2. **Packed flat arrays**: 1 array numpy grande + offsets (5.8GB vs 21GB com arrays individuais)

In [ ]:
# Parsing direto de FEN sem chess.Board
_PIECE_MAP = {
    'P': (0, 0), 'N': (0, 1), 'B': (0, 2), 'R': (0, 3), 'Q': (0, 4), 'K': (0, 5),
    'p': (1, 0), 'n': (1, 1), 'b': (1, 2), 'r': (1, 3), 'q': (1, 4), 'k': (1, 5),
}

def fen_to_sparse_fast(fen_str):
    if isinstance(fen_str, bytes): fen_str = fen_str.decode()
    parts = fen_str.split(' ')
    stm = 0 if parts[1] == 'w' else 1
    wi, bi = [], []
    sq = 56
    for ch in parts[0]:
        if ch == '/': sq -= 16
        elif ch.isdigit(): sq += int(ch)
        else:
            color, pt = _PIECE_MAP[ch]
            wi.append(color * 384 + pt * 64 + sq)
            bi.append((1 - color) * 384 + pt * 64 + (sq ^ 56))
            sq += 1
    return stm, wi, bi


class NnueSparseDatasetV2(Dataset):
    """Dataset com packed flat arrays int16 + mirror. Minimo de RAM."""

    def __init__(self, h5_path, split='train', max_pos=None, mirror=False):
        import gc
        self.mirror = mirror

        with h5py.File(h5_path, 'r') as f:
            g = f[split]
            total = len(g['evals'])
            n = min(total, max_pos) if max_pos else total
            self.evals = g['evals'][:n].astype(np.float32)
            self.wdl = g['wdl'][:n].astype(np.float32)

        self.n = n
        print(f'[{split}] {self.n:,}' + (f' (x2 mirror = {self.n*2:,})' if mirror else ''))

        # int16 em vez de int64 = 4x menos RAM (2 bytes vs 8 bytes por indice)
        # 44M pos * 28 pecas * 2 bytes * 2 = ~5GB (vs ~20GB com int64)
        max_pieces = self.n * 32
        w_flat = np.empty(max_pieces, dtype=np.int16)
        b_flat = np.empty(max_pieces, dtype=np.int16)
        offsets = np.empty(self.n + 1, dtype=np.int32)  # int32 basta (max ~1.4B < 2^31)
        self.stm = np.empty(self.n, dtype=np.uint8)

        CHUNK = 500_000
        pos = 0
        offsets[0] = 0

        for chunk_start in tqdm(range(0, self.n, CHUNK), desc=f'  {split}',
                                total=(self.n + CHUNK - 1) // CHUNK):
            chunk_end = min(chunk_start + CHUNK, self.n)
            with h5py.File(h5_path, 'r') as f:
                fens_chunk = f[split]['fens'][chunk_start:chunk_end]
            for j, raw_fen in enumerate(fens_chunk):
                i = chunk_start + j
                s, wi, bi = fen_to_sparse_fast(raw_fen)
                k = len(wi)
                w_flat[pos:pos + k] = wi
                b_flat[pos:pos + k] = bi
                pos += k
                offsets[i + 1] = pos
                self.stm[i] = s
            del fens_chunk
            gc.collect()

        self.w_flat = w_flat[:pos].copy()
        self.b_flat = b_flat[:pos].copy()
        self.offsets = offsets
        del w_flat, b_flat
        gc.collect()

        mem_mb = (self.w_flat.nbytes + self.b_flat.nbytes + self.offsets.nbytes +
                  self.evals.nbytes + self.wdl.nbytes + self.stm.nbytes) / 1e6
        print(f'  {mem_mb:.0f} MB RAM')

    def __len__(self):
        return self.n * 2 if self.mirror else self.n

    def __getitem__(self, idx):
        do_mirror = self.mirror and idx >= self.n
        real_idx = idx - self.n if do_mirror else idx

        start = self.offsets[real_idx]
        end = self.offsets[real_idx + 1]

        # Copiar e converter para int64 (torch LongTensor precisa)
        w_idx = self.w_flat[start:end].astype(np.int64)
        b_idx = self.b_flat[start:end].astype(np.int64)

        if do_mirror:
            w_idx ^= 7
            b_idx ^= 7

        stm = self.stm[real_idx]
        ev = self.evals[real_idx]
        wdl = self.wdl[real_idx]

        if stm == 1:
            ev, wdl = -ev, -wdl
            w_idx, b_idx = b_idx, w_idx

        return w_idx, b_idx, ev, wdl


def nnue_collate(batch):
    """Concatena indices esparsos e cria offsets para EmbeddingBag."""
    all_w, all_b = [], []
    w_off, b_off = [], []
    evs, wdls = [], []
    w_pos, b_pos = 0, 0

    for w_idx, b_idx, ev, wdl in batch:
        all_w.append(w_idx)
        all_b.append(b_idx)
        w_off.append(w_pos)
        b_off.append(b_pos)
        w_pos += len(w_idx)
        b_pos += len(b_idx)
        evs.append(ev)
        wdls.append(wdl)

    return (
        torch.from_numpy(np.concatenate(all_w)),
        torch.tensor(w_off, dtype=torch.long),
        torch.from_numpy(np.concatenate(all_b)),
        torch.tensor(b_off, dtype=torch.long),
        torch.tensor(evs, dtype=torch.float32),
        torch.tensor(wdls, dtype=torch.float32),
    )


print('OK')

---
## Modelo NNUE v2 (768→256→8→1)

In [ ]:
class ClippedReLU(nn.Module):
    def forward(self, x): return torch.clamp(x, 0.0, 1.0)

class PelancaNNUEv2(nn.Module):
    def __init__(self):
        super().__init__()
        self.ft = nn.EmbeddingBag(INPUT_SIZE, FT_SIZE, mode='sum', sparse=False)
        self.ft_bias = nn.Parameter(torch.zeros(FT_SIZE))
        self.hidden = nn.Linear(FT_SIZE * 2, HIDDEN_SIZE)
        self.out = nn.Linear(HIDDEN_SIZE, 1)
        self.crelu = ClippedReLU()
        self._init()

    def _init(self):
        nn.init.kaiming_normal_(self.ft.weight, nonlinearity='relu')
        nn.init.zeros_(self.ft_bias)
        nn.init.kaiming_normal_(self.hidden.weight, nonlinearity='relu')
        nn.init.zeros_(self.hidden.bias)
        nn.init.xavier_normal_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, stm_idx, stm_off, nstm_idx, nstm_off):
        # EmbeddingBag FORA do autocast (precisa FP32 na T4)
        with torch.amp.autocast('cuda', enabled=False):
            stm_acc = self.crelu(self.ft(stm_idx, stm_off) + self.ft_bias)
            nstm_acc = self.crelu(self.ft(nstm_idx, nstm_off) + self.ft_bias)
        # Hidden + output podem usar AMP
        h = self.crelu(self.hidden(torch.cat([stm_acc, nstm_acc], dim=1)))
        return self.out(h)


def wdl_blended_loss(pred, target_eval, target_wdl, wdl_w=WDL_WEIGHT):
    pred_sig = torch.tanh(pred.float())
    return (1.0 - wdl_w) * F.mse_loss(pred_sig, target_eval) + \
           wdl_w * F.mse_loss(pred_sig, target_wdl)

m = PelancaNNUEv2()
print(f'Params: {sum(p.numel() for p in m.parameters()):,}')
del m

---
## Export (formato binario v2)

In [ ]:
def export_nnue_v2(model, path, verbose=True):
    model.eval()
    m = model.module if hasattr(model, 'module') else model
    with open(path, 'wb') as f:
        f.write(b'PLNN')
        f.write(struct.pack('<I', 2))
        f.write(struct.pack('<I', INPUT_SIZE))
        f.write(struct.pack('<I', FT_SIZE))
        f.write(struct.pack('<I', HIDDEN_SIZE))

        # EmbeddingBag.weight: [768, 256] → transpor para [256, 768] (formato do Rust)
        ft_w = (m.ft.weight.data.cpu().T * FT_QUANT).round().clamp(-32767, 32767).to(torch.int16)
        ft_b = (m.ft_bias.data.cpu() * FT_QUANT).round().clamp(-32767, 32767).to(torch.int16)
        f.write(ft_w.contiguous().numpy().tobytes())
        f.write(ft_b.numpy().tobytes())

        h_w = (m.hidden.weight.data.cpu() * HIDDEN_QUANT).round().clamp(-127, 127).to(torch.int8)
        h_b = (m.hidden.bias.data.cpu() * FT_QUANT * HIDDEN_QUANT).round().clamp(-2**30, 2**30).to(torch.int32)
        f.write(h_w.numpy().tobytes())
        f.write(h_b.numpy().tobytes())

        o_w = (m.out.weight.data.cpu() * OUTPUT_QUANT).round().clamp(-127, 127).to(torch.int8)
        o_b = (m.out.bias.data.cpu() * FT_QUANT * HIDDEN_QUANT * OUTPUT_QUANT).round().clamp(-2**30, 2**30).to(torch.int32)
        f.write(o_w.numpy().tobytes())
        f.write(o_b.numpy().tobytes())

    if verbose:
        sz = os.path.getsize(path)
        print(f'Exportado: {path} ({sz:,} bytes / {sz/1024:.1f} KB)')

print('Export OK')

---
## Carregar Dataset

In [ ]:
import gc; gc.collect()
print('Carregando TODOS os dados + mirror augmentation...\n')

train_ds = NnueSparseDatasetV2(HDF5_FILE, 'train', mirror=MIRROR_AUG)
gc.collect()
val_ds = NnueSparseDatasetV2(HDF5_FILE, 'val', max_pos=500_000)
gc.collect()


def make_batches(ds, batch_size, shuffle=True):
    """Gerador de batches 100% VECTORIZADO com numpy.
    Zero loops Python por sample. ~10x mais rapido que DataLoader."""
    n = len(ds)
    perm = np.random.permutation(n) if shuffle else np.arange(n)

    for batch_start in range(0, n - batch_size + 1, batch_size):
        idx = perm[batch_start:batch_start + batch_size]

        # Separar indices reais e mirror
        mirror_mask = idx >= ds.n
        real_idx = np.where(mirror_mask, idx - ds.n, idx)

        B = len(idx)
        starts = ds.offsets[real_idx].astype(np.int64)
        ends = ds.offsets[real_idx + 1].astype(np.int64)
        lengths = ends - starts

        cumlen = np.cumsum(lengths)
        batch_off = np.empty(B, dtype=np.int64)
        batch_off[0] = 0
        batch_off[1:] = cumlen[:-1]
        total = int(cumlen[-1])

        # Construir indices flat VECTORIZADO (np.repeat + arange, zero loop)
        base = np.repeat(starts, lengths)
        within = np.arange(total, dtype=np.int64) - np.repeat(batch_off, lengths)
        flat = base + within

        # Lookup nos packed arrays (1 operacao numpy)
        all_w = ds.w_flat[flat].astype(np.int64)
        all_b = ds.b_flat[flat].astype(np.int64)

        # Mirror: XOR 7 (vectorizado)
        if ds.mirror and mirror_mask.any():
            m_exp = np.repeat(mirror_mask, lengths)
            all_w[m_exp] ^= 7
            all_b[m_exp] ^= 7

        # STM: swap w/b para pretas + negar eval (vectorizado)
        black = ds.stm[real_idx] == 1
        evs = ds.evals[real_idx].copy()
        wdls = ds.wdl[real_idx].copy()
        evs[black] *= -1
        wdls[black] *= -1

        if black.any():
            b_exp = np.repeat(black, lengths)
            tmp = all_w[b_exp].copy()
            all_w[b_exp] = all_b[b_exp]
            all_b[b_exp] = tmp

        yield (
            torch.from_numpy(all_w), torch.from_numpy(batch_off),
            torch.from_numpy(all_b), torch.from_numpy(batch_off.copy()),
            torch.from_numpy(evs), torch.from_numpy(wdls),
        )


# Teste rapido: criar 1 batch e medir tempo
t_test = time.time()
for _ in make_batches(train_ds, BATCH_SIZE):
    break
t_batch = time.time() - t_test
print(f'1 batch ({BATCH_SIZE:,} samples) criado em {t_batch:.2f}s')

effective = len(train_ds)
n_batches = effective // BATCH_SIZE
est_epoch = t_batch * n_batches
print(f'Train: {n_batches} batches | Est. epoch: {est_epoch:.0f}s')
print(f'Ratio: {effective // params}:1')

import psutil
ram = psutil.virtual_memory()
print(f'RAM: {ram.used/1e9:.1f} / {ram.total/1e9:.1f} GB ({ram.percent}%)')

---
## Treinar (60 epochs, cosine LR)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = torch.cuda.is_available()

model = PelancaNNUEv2().to(device)

optimizer = optim.Adam(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
scaler = torch.amp.GradScaler('cuda') if use_amp else None

best_val = float('inf')
hist = {'train': [], 'val': [], 'lr': []}
n_train_batches = len(train_ds) // BATCH_SIZE
n_val_batches = max(1, len(val_ds) // BATCH_SIZE)

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Treino: {EPOCHS} epochs | LR: {LR_MAX} -> {LR_MIN} (cosine)')
print(f'Loss: {1-WDL_WEIGHT:.0%} eval + {WDL_WEIGHT:.0%} WDL')
print(f'Batches/epoch: {n_train_batches} (vectorizado, sem DataLoader)\n')

for epoch in range(EPOCHS):
    t0 = time.time()

    # Train (batches vectorizados)
    model.train()
    tl, tn = 0.0, 0
    for stm_idx, stm_off, nstm_idx, nstm_off, ev, wdl in make_batches(train_ds, BATCH_SIZE, shuffle=True):
        stm_idx = stm_idx.to(device, non_blocking=True)
        stm_off = stm_off.to(device, non_blocking=True)
        nstm_idx = nstm_idx.to(device, non_blocking=True)
        nstm_off = nstm_off.to(device, non_blocking=True)
        ev = ev.to(device, non_blocking=True).unsqueeze(1)
        wdl = wdl.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.amp.autocast('cuda'):
                pred = model(stm_idx, stm_off, nstm_idx, nstm_off)
                loss = wdl_blended_loss(pred, ev, wdl)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(stm_idx, stm_off, nstm_idx, nstm_off)
            loss = wdl_blended_loss(pred, ev, wdl)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        tl += loss.item(); tn += 1

    train_loss = tl / max(tn, 1)

    # Val
    model.eval()
    vl, vn = 0.0, 0
    with torch.no_grad():
        for stm_idx, stm_off, nstm_idx, nstm_off, ev, wdl in make_batches(val_ds, BATCH_SIZE, shuffle=False):
            stm_idx = stm_idx.to(device)
            stm_off = stm_off.to(device)
            nstm_idx = nstm_idx.to(device)
            nstm_off = nstm_off.to(device)
            ev = ev.to(device).unsqueeze(1)
            wdl = wdl.to(device).unsqueeze(1)
            pred = model(stm_idx, stm_off, nstm_idx, nstm_off)
            loss = wdl_blended_loss(pred, ev, wdl)
            vl += loss.item(); vn += 1

    val_loss = vl / max(vn, 1)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    hist['train'].append(train_loss)
    hist['val'].append(val_loss)
    hist['lr'].append(lr)

    mk = ''
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'epoch': epoch, 'model': model.state_dict(), 'val': val_loss},
                   os.path.join(CKPT_DIR, 'best.pt'))
        mk = ' ** BEST'

    elapsed = time.time() - t0
    print(f'E{epoch:3d} | t={train_loss:.6f} v={val_loss:.6f} lr={lr:.1e} | {elapsed:.0f}s{mk}')

    if (epoch + 1) % 15 == 0:
        p = os.path.join(CKPT_DIR, f'e{epoch}.nnue')
        export_nnue_v2(model, p, verbose=False)
        print(f'  -> {p}')

print(f'\nMelhor val: {best_val:.6f}')

---
## Grafico

In [ ]:
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.plot(hist['train'], label='Train'); a1.plot(hist['val'], label='Val')
a1.set_xlabel('Epoch'); a1.set_ylabel('Loss'); a1.legend(); a1.grid(alpha=0.3)
a1.set_title('Loss')
a2.plot(hist['lr']); a2.set_xlabel('Epoch'); a2.set_ylabel('LR')
a2.set_title('Learning Rate (Cosine)'); a2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/loss_v2.png', dpi=150); plt.show()

---
## Exportar Modelo Final

In [ ]:
ck = torch.load(os.path.join(CKPT_DIR, 'best.pt'), map_location='cpu')
fm = PelancaNNUEv2()
fm.load_state_dict(ck['model'])
print(f'Best: epoch {ck["epoch"]}, val={ck["val"]:.6f}\n')
export_nnue_v2(fm, NNUE_OUTPUT)

print(f'\n{"="*60}')
print(f'PRONTO: {NNUE_OUTPUT}')
print(f'Tamanho: {os.path.getsize(NNUE_OUTPUT):,} bytes')
print(f'\n1. Baixe pelanca_v2.nnue')
print(f'2. Renomeie para pelanca.nnue')
print(f'3. Coloque em nn/pelanca.nnue no projeto')
print(f'4. cargo build --release')
print(f'{"="*60}')

---
## Teste

In [ ]:
fm.eval()
tests = [
    ('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1', 'Inicial'),
    ('rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1', '1.e4'),
    ('8/8/8/8/8/8/4K3/R3k3 w - - 0 1', 'KR vs K'),
    ('8/8/8/8/8/8/4k3/r3K3 b - - 0 1', 'kr vs K'),
    ('r1bqkbnr/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3', 'Ruy Lopez'),
    ('rnbqkb1r/pppp1ppp/5n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3', 'Italian'),
]

print(f'{"Pos":<12} {"Raw":>8} {"tanh":>8} {"~cp":>8}')
print('-' * 45)
for fen, desc in tests:
    s, wi, bi = fen_to_sparse_fast(fen)
    if s == 1:
        wi, bi = bi, wi

    stm_idx = torch.tensor(wi, dtype=torch.long).unsqueeze(0).to('cpu')
    stm_off = torch.tensor([0], dtype=torch.long)
    nstm_idx = torch.tensor(bi, dtype=torch.long).unsqueeze(0).to('cpu')
    nstm_off = torch.tensor([0], dtype=torch.long)

    # Flatten for EmbeddingBag
    stm_idx = stm_idx.squeeze(0)
    nstm_idx = nstm_idx.squeeze(0)

    with torch.no_grad():
        r = fm(stm_idx, stm_off, nstm_idx, nstm_off).item()
    t = np.tanh(r)
    cp = np.arctanh(np.clip(t, -0.9999, 0.9999)) * EVAL_SCALE
    print(f'{desc:<12} {r:>+8.3f} {t:>+8.3f} {cp:>+8.0f}')